<a href="https://colab.research.google.com/github/LucasMoraes023/atlas-image-ai/blob/main/colab/ATLAS_Image_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ATLAS IMAGE AI (Colab)
Gerador de imagens com SDXL na GPU gratuita do Google Colab.

1. **Ambiente de execução (Runtime) → Alterar o tipo de ambiente (Change runtime type) → T4 GPU** → Salvar.
2. **Ambiente de execução → Executar tudo (Run all)**. Na primeira vez leva alguns minutos, porque baixa cerca de 7 GB.
3. No fim da última célula aparece um link `https://....gradio.live`. Abra esse link no celular.

**Privacidade:** prompts e imagens passam pelos servidores do Google (Colab) e pelo relé do link público do Gradio.
**Licenças:** SDXL 1.0 base usa a CreativeML Open RAIL++-M; no SDXL-Turbo, confira a licença atual no cartão do modelo (pode ser só uso não comercial). As restrições de uso dos modelos continuam valendo.
Este app não adiciona filtros e não altera o prompt: o texto enviado é exatamente o do campo. Prompts acima de 77 tokens são cortados pelo modelo.

In [ ]:
!pip -q install diffusers transformers accelerate safetensors gradio
import torch
assert torch.cuda.is_available(), "Sem GPU. Ative em Ambiente de execução → Alterar o tipo de ambiente → T4 GPU e execute de novo."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import gc, os, random, time
import torch, gradio as gr
from PIL.PngImagePlugin import PngInfo
from diffusers import AutoPipelineForText2Image

MODELS = {
    "SDXL 1.0 base (qualidade)": {"repo": "stabilityai/stable-diffusion-xl-base-1.0", "turbo": False},
    "SDXL-Turbo (rápido, 512 px)": {"repo": "stabilityai/sdxl-turbo", "turbo": True},
}
SIZES = {
    "1:1 · 1024×1024": (1024, 1024), "3:4 · 896×1152": (896, 1152), "4:3 · 1152×896": (1152, 896),
    "9:16 · 768×1344": (768, 1344), "16:9 · 1344×768": (1344, 768),
}
PRESETS = {
    "Fotografia": "photo, 35mm, natural light, sharp focus, high detail",
    "Retrato": "portrait, soft studio light, shallow depth of field, detailed face",
    "Paisagem": "wide landscape, golden hour, dramatic sky, high detail",
    "Anime": "anime style, clean line art, vibrant colors",
    "Ilustração": "digital illustration, bold shapes, painterly, detailed",
    "Fantasia": "fantasy art, epic, magical atmosphere, intricate details",
    "Arte conceitual": "concept art, cinematic composition, matte painting",
    "Render 3D": "3d render, octane render, global illumination, ultra detailed",
}
OUT = "/content/atlas_outputs"
os.makedirs(OUT, exist_ok=True)
cur = {"name": None, "pipe": None}
stop = {"flag": False}

def get_pipe(name):
    if cur["name"] == name:
        return cur["pipe"]
    cur["pipe"], cur["name"] = None, None
    gc.collect(); torch.cuda.empty_cache()
    pipe = AutoPipelineForText2Image.from_pretrained(
        MODELS[name]["repo"], torch_dtype=torch.float16, variant="fp16", use_safetensors=True).to("cuda")
    cur["pipe"], cur["name"] = pipe, name
    return pipe

def generate(model, prompt, negative, size, steps, cfg, seed, count, progress=gr.Progress()):
    prompt = (prompt or "").strip()
    if not prompt:
        raise gr.Error("Escreva um prompt.")
    stop["flag"] = False
    progress(0, desc="Carregando o modelo…")
    pipe = get_pipe(model)
    w, h = SIZES[size]
    steps, cfg, negative, count = int(steps), float(cfg), (negative or "").strip(), int(count)
    notes = []
    if MODELS[model]["turbo"]:
        w, h = w // 2, h // 2
        if cfg != 0 or negative or steps > 4:
            notes.append("SDXL-Turbo: orientação 0, sem prompt negativo e no máximo 4 etapas (valores ajustados).")
        cfg, negative, steps = 0.0, "", min(steps, 4)
    base = random.randint(0, 2**32 - 1) if seed is None or int(seed) < 0 else int(seed)
    shown, lines, t0 = [], [], time.time()
    try:
        for i in range(count):
            s = (base + i) % 2**32
            def tick(p, step, t, kw, i=i):
                progress((step + 1, steps), desc=f"Imagem {i + 1}/{count}")
                return kw
            img = pipe(prompt=prompt, negative_prompt=negative or None, width=w, height=h,
                       num_inference_steps=steps, guidance_scale=cfg,
                       generator=torch.Generator("cuda").manual_seed(s), callback_on_step_end=tick).images[0]
            if stop["flag"]:
                notes.append("Interrompido pelo usuário.")
                break
            desc = f"prompt: {prompt}\nnegativo: {negative}\nmodelo: {model} | {w}x{h} | etapas {steps} | CFG {cfg} | seed {s}"
            meta = PngInfo(); meta.add_text("parameters", desc)
            img.save(f"{OUT}/atlas_{int(time.time())}_{s}.png", pnginfo=meta)
            shown.append((img, f"seed {s}")); lines.append(desc)
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        raise gr.Error("Memória da GPU insuficiente. Use uma resolução menor ou o SDXL-Turbo.")
    return shown, "\n\n".join(lines + notes) + f"\n\nTempo: {time.time() - t0:.1f} s"

def do_stop():
    stop["flag"] = True
    if cur["pipe"] is not None:
        cur["pipe"]._interrupt = True

def add_preset(p, k):
    p = (p or "").strip()
    return p if not k else (f"{p}, {PRESETS[k]}" if p else PRESETS[k])

print("Carregando o modelo padrão (pode demorar alguns minutos)…")
get_pipe(list(MODELS)[0])

with gr.Blocks(title="ATLAS IMAGE AI", theme=gr.themes.Soft(primary_hue="violet", secondary_hue="blue")) as demo:
    gr.Markdown("# ATLAS IMAGE AI\nO prompt é enviado ao modelo exatamente como está no campo.")
    model = gr.Dropdown(list(MODELS), value=list(MODELS)[0], label="Modelo")
    prompt = gr.Textbox(label="Prompt", lines=4)
    with gr.Row():
        preset = gr.Dropdown(list(PRESETS), value=None, label="Preset (acrescenta texto ao prompt)")
        add = gr.Button("Aplicar preset")
    negative = gr.Textbox(label="Prompt negativo", lines=2)
    size = gr.Dropdown(list(SIZES), value=list(SIZES)[0], label="Proporção e resolução")
    with gr.Row():
        steps = gr.Slider(1, 50, value=30, step=1, label="Etapas")
        cfg = gr.Slider(0, 15, value=7, step=0.5, label="Orientação (CFG)")
    with gr.Row():
        seed = gr.Number(value=-1, precision=0, label="Seed (-1 = aleatória)")
        count = gr.Slider(1, 4, value=1, step=1, label="Quantidade")
    with gr.Row():
        go = gr.Button("Gerar", variant="primary")
        halt = gr.Button("Interromper")
    gallery = gr.Gallery(label="Resultados", columns=2)
    info = gr.Textbox(label="O que foi processado", lines=6)
    add.click(add_preset, [prompt, preset], prompt)
    go.click(generate, [model, prompt, negative, size, steps, cfg, seed, count], [gallery, info])
    halt.click(do_stop, None, None)

demo.queue().launch(share=True, debug=True)

Carregando o modelo padrão (pode demorar alguns minutos)…


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

/tmp/ipykernel_673/2899607184.py:89: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="ATLAS IMAGE AI", theme=gr.themes.Soft(primary_hue="violet", secondary_hue="blue")) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://bd5ec2e8a8a60f63fc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/diffusers/configuration_utils.py:172: FutureWarning: Accessing config attribute `__iter__` directly via 'StableDiffusionXLPipeline' object attribute is deprecated. Please access '__iter__' over 'StableDiffusionXLPipeline's config object instead, e.g. 'scheduler.config.__iter__'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)
/usr/local/lib/python3.13/dist-packages/diffusers/configuration_utils.py:172: FutureWarning: Accessing config attribute `__iter__` directly via 'StableDiffusionXLPipeline' object attribute is deprecated. Please access '__iter__' over 'StableDiffusionXLPipeline's config object instead, e.g. 'scheduler.config.__iter__'.
  deprecate("direct config name access", "1.0.0", deprecation_message, standard_warn=False)
/usr/local/lib/python3.13/dist-packages/diffusers/configuration_utils.py:172: FutureWarning: Accessing config attribute `__iter__` directly via 'StableDiffusionXLPipeline' obje